# Boosted Trees Health Condition Classifier

This notebook trains boosted-tree classifiers on the leakage-safe numeric feature files. It compares unweighted and class-balanced variants on the validation split, chooses the best validation setup, retrains on all labeled data, and writes a submission file.

## 1. Load Numeric Features

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"

train_df = pd.read_csv("data/train_split_features_numeric.csv")
val_df = pd.read_csv("data/val_split_features_numeric.csv")
test_df = pd.read_csv("data/test_features_numeric.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)
print("sample submission:", sample_submission.shape)

train: (552070, 73)
validation: (138018, 73)
test: (295753, 72)
sample submission: (295753, 2)


## 2. Prepare X/y/test

In [3]:
feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]

assert [col for col in train_df.columns if col != TARGET_COL] == list(val_df.drop(columns=[TARGET_COL]).columns)
assert [col for col in train_df.columns if col != TARGET_COL] == list(test_df.columns)

X_train = train_df[feature_cols].astype("float32")
y_train_raw = train_df[TARGET_COL]
X_val = val_df[feature_cols].astype("float32")
y_val_raw = val_df[TARGET_COL]
X_test = test_df[feature_cols].astype("float32")
test_ids = test_df[ID_COL].copy()

print("feature count:", len(feature_cols))
print("missing train:", int(X_train.isna().sum().sum()))
print("missing val:", int(X_val.isna().sum().sum()))
print("missing test:", int(X_test.isna().sum().sum()))

feature count: 71
missing train: 0
missing val: 0
missing test: 0


## 3. Target Encoding

In [4]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_

label_mapping = pd.DataFrame({
    "class_id": np.arange(len(class_names)),
    "health_condition": class_names,
})
label_mapping

,class_id,health_condition
0,0,at-risk
1,1,fit
2,2,unhealthy


## 4. Baseline Majority Class

In [5]:
majority_label = y_train_raw.mode()[0]
majority_pred = np.full(len(y_val_raw), majority_label)

baseline_report = pd.DataFrame([
    {
        "model": "majority_class",
        "validation_accuracy": accuracy_score(y_val_raw, majority_pred),
        "validation_macro_f1": f1_score(y_val_raw, majority_pred, average="macro"),
        "validation_weighted_f1": f1_score(y_val_raw, majority_pred, average="weighted"),
    }
])
baseline_report

,model,validation_accuracy,validation_macro_f1,validation_weighted_f1
0,majority_class,0.858671,0.307987,0.793379


## 5. Boosted Tree Experiments

Two variants are tested:

- `hist_gb_unweighted`: usually best if leaderboard score is accuracy.
- `hist_gb_balanced`: can improve minority classes, but may lower accuracy.

In [6]:
experiment_configs = [
    {
        "name": "hist_gb_unweighted",
        "class_weight": None,
        "learning_rate": 0.06,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.01,
    },
    {
        "name": "hist_gb_balanced",
        "class_weight": "balanced",
        "learning_rate": 0.06,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.01,
    },
]

experiment_results = []
trained_models = {}

for config in experiment_configs:
    print("Training", config["name"])
    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=config["learning_rate"],
        max_iter=config["max_iter"],
        max_leaf_nodes=config["max_leaf_nodes"],
        l2_regularization=config["l2_regularization"],
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
        class_weight=config["class_weight"],
        verbose=0,
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_pred_labels = label_encoder.inverse_transform(val_pred)

    result = {
        "model": config["name"],
        "validation_accuracy": accuracy_score(y_val_raw, val_pred_labels),
        "validation_macro_f1": f1_score(y_val_raw, val_pred_labels, average="macro"),
        "validation_weighted_f1": f1_score(y_val_raw, val_pred_labels, average="weighted"),
        "n_iter": model.n_iter_,
        "class_weight": config["class_weight"],
    }
    experiment_results.append(result)
    trained_models[config["name"]] = model
    print(result)

experiment_report = pd.DataFrame(experiment_results).sort_values("validation_accuracy", ascending=False).reset_index(drop=True)
experiment_report

Training hist_gb_unweighted


{'model': 'hist_gb_unweighted', 'validation_accuracy': 0.9661710791346056, 'validation_macro_f1': 0.9059949966617421, 'validation_weighted_f1': 0.9647650725154651, 'n_iter': 136, 'class_weight': None}
Training hist_gb_balanced


{'model': 'hist_gb_balanced', 'validation_accuracy': 0.8826240055644916, 'validation_macro_f1': 0.7668287722426963, 'validation_weighted_f1': 0.894358384366474, 'n_iter': 123, 'class_weight': 'balanced'}


,model,validation_accuracy,validation_macro_f1,validation_weighted_f1,n_iter,class_weight
0,hist_gb_unweighted,0.966171,0.905995,0.964765,136,None
1,hist_gb_balanced,0.882624,0.766829,0.894358,123,balanced


## 6. Select Best Model

For the Kaggle-style submission, select by validation accuracy. Macro F1 is still shown so we can see minority-class behavior.

In [7]:
best_model_name = experiment_report.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("best model by validation accuracy:", best_model_name)
display(experiment_report)

best model by validation accuracy: hist_gb_unweighted


,model,validation_accuracy,validation_macro_f1,validation_weighted_f1,n_iter,class_weight
0,hist_gb_unweighted,0.966171,0.905995,0.964765,136,None
1,hist_gb_balanced,0.882624,0.766829,0.894358,123,balanced


## 7. Evaluation

In [8]:
best_val_pred = best_model.predict(X_val)
best_val_pred_labels = label_encoder.inverse_transform(best_val_pred)

evaluation_summary = pd.DataFrame([
    {
        "model": best_model_name,
        "validation_accuracy": accuracy_score(y_val_raw, best_val_pred_labels),
        "validation_macro_f1": f1_score(y_val_raw, best_val_pred_labels, average="macro"),
        "validation_weighted_f1": f1_score(y_val_raw, best_val_pred_labels, average="weighted"),
    }
])
evaluation_summary

,model,validation_accuracy,validation_macro_f1,validation_weighted_f1
0,hist_gb_unweighted,0.966171,0.905995,0.964765


In [9]:
print(classification_report(y_val_raw, best_val_pred_labels, labels=class_names))

              precision    recall  f1-score   support

     at-risk       0.97      0.99      0.98    118512
         fit       0.95      0.81      0.87      7961
   unhealthy       0.97      0.78      0.86     11545

    accuracy                           0.97    138018
   macro avg       0.96      0.86      0.91    138018
weighted avg       0.97      0.97      0.96    138018



In [10]:
confusion = pd.DataFrame(
    confusion_matrix(y_val_raw, best_val_pred_labels, labels=class_names),
    index=[f"actual_{label}" for label in class_names],
    columns=[f"pred_{label}" for label in class_names],
)
confusion

,pred_at-risk,pred_fit,pred_unhealthy
actual_at-risk,117901,361,250
actual_fit,1467,6468,26
actual_unhealthy,2551,14,8980


## 8. Final Full-Data Training

Retrain the selected configuration on train + validation labels before predicting the real test file.

In [11]:
best_config = next(config for config in experiment_configs if config["name"] == best_model_name)
full_train_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_train_df[feature_cols].astype("float32")
y_full = label_encoder.transform(full_train_df[TARGET_COL])

# Use the validation-selected iteration count as the final iteration budget.
# This avoids training much longer than the model needed during validation.
final_max_iter = int(best_model.n_iter_)

final_model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=best_config["learning_rate"],
    max_iter=final_max_iter,
    max_leaf_nodes=best_config["max_leaf_nodes"],
    l2_regularization=best_config["l2_regularization"],
    early_stopping=False,
    random_state=RANDOM_STATE,
    class_weight=best_config["class_weight"],
    verbose=0,
)

print("full training rows:", X_full.shape[0])
print("final max_iter:", final_max_iter)
print("final class_weight:", best_config["class_weight"])
final_model.fit(X_full, y_full)

full training rows: 690088
final max_iter: 136
final class_weight: None


HistGradientBoostingClassifier(early_stopping=False, l2_regularization=0.01,
                               learning_rate=0.06, max_iter=136,
                               random_state=42)

## 9. Test Prediction + Submission File

In [12]:
test_pred = final_model.predict(X_test)
test_pred_labels = label_encoder.inverse_transform(test_pred)

submission = sample_submission.copy()
submission[ID_COL] = test_ids.values
submission[TARGET_COL] = test_pred_labels

submission_path = "data/submission_boosted_trees.csv"
submission.to_csv(submission_path, index=False)

print("saved:", submission_path, submission.shape)
submission[TARGET_COL].value_counts(normalize=True).mul(100).round(2)

saved: data/submission_boosted_trees.csv (295753, 2)


at-risk      88.49
unhealthy     6.57
fit           4.94
Name: health_condition, dtype: float64

In [13]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
